In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

NUM_CLASSES = 10
EPOCHS = 10
LR = 1e-3

In [ ]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # נרמול ל-[0,1]
    return image, label

(ds_train, ds_test), ds_info = tfds.load(
    "cifar10",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

ds_train = (ds_train
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .shuffle(10_000)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

ds_test = (ds_test
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print("Train samples:", ds_info.splits["train"].num_examples)
print("Test samples:", ds_info.splits["test"].num_examples)

<div dir="rtl" style="text-align:right;">

#### תא 2 — טעינת CIFAR-10 + עיבוד מקדים (Preprocessing)

בתא הזה:
- מגדירים פונקציה `preprocess` שמבצעת:
  - המרה ל־`float32`
  - נרמול ערכי פיקסלים לטווח `[0,1]` על־ידי חלוקה ב־255.
  - החזרה של `(image, label)` כדי שהמודל יקבל גם קלט וגם תווית.
- טוענים את הדאטה CIFAR-10 דרך `tfds.load` ומקבלים `train` ו־`test`.
- מכינים את הדאטה לאימון:
  - `shuffle` (רק ל־train) כדי לערבב דוגמאות.
  - `batch` כדי לעבוד בקבוצות דוגמאות.
  - `prefetch` כדי לזרז את זרימת הדאטה בזמן האימון.

בסוף התא מדפיסים את מספר הדוגמאות ב־train וב־test.

</div>

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.1),
])

<div dir="rtl" style="text-align:right;">

#### תא 3 — הגדלת נתונים (Data Augmentation)

בתא הזה:
- מגדירים שכבות שמבצעות שינויי תמונה אקראיים בזמן אימון:
  - היפוך אופקי (RandomFlip)
  - הזזה קטנה (RandomTranslation)
  - סיבוב קטן (RandomRotation)

מטרת ה־augmentation:
- לגרום למודל ללמוד תבניות יציבות יותר ולא “לזכור” תמונות ספציפיות.
- לשפר הכללה (Generalization) ולהפחית Overfitting.

</div>

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

<div dir="rtl" style="text-align:right;">

#### תא 4 — ארכיטקטורת המודל

נבנתה רשת CNN עם שלוש שכבות קונבולוציה (32, 64, 128 פילטרים) וכל אחת מלווה ב-MaxPooling.
לאחר מכן מתבצע Flatten ושכבה מלאה בגודל 128 לפני שכבת Softmax ל-10 מחלקות.
למודל יש 160,202 פרמטרים ניתנים לאימון.

</div>

In [ ]:
history = model.fit(
    ds_train,
    validation_data=ds_test,
    epochs=EPOCHS
)

<div dir="rtl" style="text-align:right;">

#### תא 5 — אימון (Training)

בתא הזה מאמנים את המודל בעזרת:
- `ds_train` כסט האימון
- `ds_test` כסט ולידציה (Validation)

הפונקציה `model.fit` מחזירה אובייקט `history` שמכיל:
- `accuracy` ו־`loss` על ה־train בכל אפוק
- `val_accuracy` ו־`val_loss` על ה־validation בכל אפוק

</div>

In [ ]:
plt.figure()
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

<div dir="rtl" style="text-align:right;">

#### תא 6 — תוצאות האימון

הדיוק עלה בהדרגה והגיע לכ-62% באימון ולכ-67% בוולידציה.
ה-Loss ירד באופן עקבי בשני הסטים לאורך האפוקים.
אין סימן ל-Overfitting והמודל מראה יכולת הכללה טובה.

</div>

In [ ]:
print("Hyperparameters:")
print("BATCH_SIZE =", BATCH_SIZE)
print("EPOCHS =", EPOCHS)
print("Optimizer = Adam")
print("Learning Rate =", LR)
print("Loss = sparse_categorical_crossentropy")
print("Architecture = Conv(32,3)-ReLU-MaxPool(2) -> Conv(64,3)-ReLU-MaxPool(2) -> Conv(128,3)-ReLU-MaxPool(2) -> Flatten -> Dense(128)-ReLU -> Dense(10)-Softmax")
print("Data Augmentation = RandomFlip(horizontal), RandomTranslation(0.1,0.1), RandomRotation(0.1)")

<div dir="rtl" style="text-align:right;">

#### תא 7 — היפר־פרמטרים

האימון בוצע עם Batch Size של 128 במשך 10 אפוקים.
נעשה שימוש באופטימייזר Adam עם קצב למידה 0.001 ופונקציית הפסד Cross-Entropy.
נוסף Data Augmentation הכולל היפוך, הזזה וסיבוב קל לשיפור ההכללה.

</div>

In [ ]:
IMG_SIZE = 224

def preprocess_tl(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

ds_train_tl = (ds_train.unbatch()
               .map(preprocess_tl, num_parallel_calls=AUTOTUNE)
               .batch(BATCH_SIZE)
               .prefetch(AUTOTUNE))

ds_test_tl = (ds_test.unbatch()
              .map(preprocess_tl, num_parallel_calls=AUTOTUNE)
              .batch(BATCH_SIZE)
              .prefetch(AUTOTUNE))

In [ ]:
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0

def build_model(base_model_class):
    base_model = base_model_class(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    
    base_model.trainable = False  # freeze
    
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model

In [ ]:
model_mobilenet = build_model(MobileNetV2)
history_mobilenet = model_mobilenet.fit(
    ds_train_tl,
    validation_data=ds_test_tl,
    epochs=5
)

In [ ]:
model_resnet = build_model(ResNet50)
history_resnet = model_resnet.fit(
    ds_train_tl,
    validation_data=ds_test_tl,
    epochs=5
)

In [ ]:
model_efficient = build_model(EfficientNetB0)
history_efficient = model_efficient.fit(
    ds_train_tl,
    validation_data=ds_test_tl,
    epochs=5
)